# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedsamymohamad/flyrank_internship_starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

**Lane 2 — Refresh / Content Opportunity Scoring**

Four things in order:
1. The contract — five plain-words answers
2. Three verification queries with visible outputs (on `month=2026-03`)
3. Five features with "knowable when?" lines, plus the deliberate-leak experiment
4. One named limitation

> ⚠️ **Before running:** request gate access at https://huggingface.co/datasets/FlyRank/internship-warehouse (instant approval), create a plain **Read** token in your HF settings, and store it as a Colab Secret named `HF_TOKEN`. Never paste the token into a cell — this repo is public.

---
## 0. Setup — install, authenticate, connect

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'huggingface_hub', 'duckdb>=0.10'], check=True)
print('Dependencies ready.')

In [ ]:
import os

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')

if not HF_TOKEN:
    raise EnvironmentError(
        'HF_TOKEN not found. Add it as a Colab Secret named HF_TOKEN '
        '(the key icon in the left sidebar). Never paste the token into a cell.'
    )
print('Token loaded (length:', len(HF_TOKEN), ')')

In [ ]:
import duckdb

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
# Use CREATE SECRET — the recommended auth method for DuckDB >= 0.10
con.execute(f"CREATE SECRET hf_secret (TYPE huggingface, TOKEN '{HF_TOKEN}');")

BASE = "hf://datasets/FlyRank/internship-warehouse"

# Mid-panel month — safe for developing label logic
# NEVER develop label logic on the _sample (June 2026 = sealed test month)
MONTH = "month=2026-03"
FACT  = f"{BASE}/fact_content_daily_performance/{MONTH}/*.parquet"
Q90D  = f"{BASE}/fact_content_query_90d/*.parquet"

# Smoke-test: count rows in the chosen partition
n = con.execute(f"SELECT COUNT(*) FROM read_parquet('{FACT}')").fetchone()[0]
print(f'Partition {MONTH}: {n:,} rows — connection OK.')

---
## 1. The Contract — five plain-words answers

### 1.1 — One row means…

One row = **one content item (page) on one calendar date for one client**.
The grain of `fact_content_daily_performance` is `(report_date, client_hash_id, content_hash_id)` — a single day's measured performance for a single page belonging to a single client.

### 1.2 — Table(s) I'll use

**Primary:** `fact_content_daily_performance`, partitioned by `month=YYYY-MM`. I develop on `month=2026-03` (a mid-panel month). I treat `month=2026-06` (the `_sample` table) as a **sealed test month** — developing label logic there means developing inside the natural outcome window of any past→future label.

**For 30-day window features:** `fact_content_query_90d` — this table pre-computes `impressions_prev30`, `clicks_prev30`, `avg_position_prev30` (days 31–60 back) and `impressions_last30`, `clicks_last30` (most recent 30 days). These are safe feature and label sources when the windows are aligned correctly.

### 1.3 — Time window

The **feature window** is the prev-30 sub-window (days 31–60 before the snapshot end): `impressions_prev30`, `clicks_prev30`, `avg_position_prev30` from `fact_content_query_90d`. The **label window** is the last-30 sub-window: decline is detected when `impressions_last30 < impressions_prev30 × 0.80`. The two windows do not overlap — that is the boundary the leakage experiment below deliberately crosses.

### 1.4 — What I predict (label / proxy)

**Proxy label:** `is_declining = 1` when impressions in the most-recent 30 days fell more than 20% compared with the prior 30 days: `impressions_last30 / impressions_prev30 < 0.80`. This is a **rule-based proxy** — not a directly observed editorial outcome. Results are labelled as directional / decision-support, not causal.

### 1.5 — One deliberate exclusion

I exclude **`impressions_last30`** and any column derived from the last-30d window (`clicks_last30`, `avg_position_last30`). These columns overlap with the label's computation window. Using them as features means the model reads the answer rather than learning from leading signals. Section 3 demonstrates this deliberately.

In [ ]:
contract = {
    '1 — One row':  'one content item on one report_date for one client '
                    '→ grain: (report_date, client_hash_id, content_hash_id)',
    '2 — Tables':   'fact_content_daily_performance (month=2026-03) for daily grain; '
                    'fact_content_query_90d for pre-aggregated 30d window features',
    '3 — Window':   'features from prev30 window (days 31-60 back); '
                    'label from last30 vs prev30 impressions comparison',
    '4 — Label':    'is_declining = 1 when impressions_last30 / impressions_prev30 < 0.80 '
                    '(rule-based proxy; not a directly observed editorial outcome)',
    '5 — Excluded': 'impressions_last30, clicks_last30, avg_position_last30 '
                    '(overlap the label window → leakage); '
                    'client_hash_id, content_hash_id (IDs — grouping/joins only, never features)',
}

print('=== DATA CONTRACT — Lane 2: Refresh / Content Opportunity Scoring ===')
for k, v in contract.items():
    print(f'\n{k}:\n  {v}')

---
## 2. Verify it — three queries with visible outputs

All three run on the mid-panel month (`month=2026-03`). A claim without a query under it is a guess.

In [ ]:
# ── Query 1: GRAIN ────────────────────────────────────────────────────────────
# Claim: one row = one (report_date, client_hash_id, content_hash_id) triple.
# If the grain holds, no triple appears more than once → zero rows returned.

import pandas as pd

q1 = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS c
FROM read_parquet('{FACT}')
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
"""

result_q1 = con.execute(q1).df()
print('Query 1 — Grain check: rows where (report_date, client_hash_id, content_hash_id) > 1')
print(f'Rows returned: {len(result_q1)}  (0 = grain holds ✓)')
if len(result_q1) > 0:
    print(result_q1)

In [ ]:
# ── Query 2: ROW COUNT + DATE SPAN ────────────────────────────────────────────
# Confirm how many rows are in month=2026-03 and the exact date range.

q2 = f"""
SELECT
    COUNT(*)                       AS total_rows,
    COUNT(DISTINCT client_hash_id) AS unique_clients,
    COUNT(DISTINCT content_hash_id)AS unique_pages,
    MIN(report_date)               AS earliest_date,
    MAX(report_date)               AS latest_date
FROM read_parquet('{FACT}')
"""

result_q2 = con.execute(q2).df()
print('Query 2 — Row count and date span for month=2026-03')
print(result_q2.T.to_string())

In [ ]:
# ── Query 3: AVAILABILITY — filter with IS TRUE ───────────────────────────────
# ga4_data_available is THREE-valued: TRUE, FALSE, or NULL.
# Rows where it is NULL carry null GA4 metrics — not zero-filled, not flagged FALSE.
# Using = TRUE silently mishandles NULLs; IS TRUE handles all three states correctly.
#
# Claim: IS TRUE filter isolates the rows with real GSC and GA4 signal.

q3 = f"""
SELECT
    COUNT(*)                                                       AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)             AS gsc_available_rows,
    ROUND(
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)
        * 100.0 / COUNT(*), 2)                                     AS gsc_available_pct,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)             AS ga4_available_rows,
    ROUND(
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)
        * 100.0 / COUNT(*), 2)                                     AS ga4_available_pct,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE
                       AND ga4_data_available IS TRUE)             AS both_available_rows
FROM read_parquet('{FACT}')
"""

result_q3 = con.execute(q3).df()
print('Query 3 — Availability: IS TRUE filter on gsc_data_available and ga4_data_available')
print(result_q3.T.to_string())
print()
print('Key: IS TRUE (not = TRUE) correctly excludes NULL rows.')
print('Only both_available_rows carry real GSC + GA4 signal safe to use as features.')

---
## 3. Five features + the leakage trap

### 3a. Feature frame

All five features come from the **prev30 window** (days 31–60 before snapshot) or from aggregated daily data. None overlaps with the label window (last30). The 30-day sub-windows come from `fact_content_query_90d`; daily counts come from `fact_content_daily_performance`.

| Feature | Source column | Available when? |
|---|---|---|
| `log_impressions_prev30` | `log1p(impressions_prev30)` from `fact_content_query_90d` | Knowable at decision moment — covers days 31–60 before snapshot, entirely before the last30 label window |
| `log_clicks_prev30` | `log1p(clicks_prev30)` from `fact_content_query_90d` | Knowable at decision moment — same prev30 window, no overlap with label |
| `avg_position_prev30` | `avg_position_prev30` from `fact_content_query_90d` | Knowable at decision moment — average GSC rank in days 31–60 back; not derived from the label window |
| `days_with_gsc` | `COUNT(DISTINCT report_date) WHERE gsc_impressions > 0` from daily fact | Knowable at decision moment — counts days with any impression in the month; measures visibility regularity |
| `gsc_avg_position` | `AVG(gsc_avg_position)` from daily fact (month=2026-03) | Knowable at decision moment — average of the daily GSC position across the whole month; not a last30 column |

In [ ]:
import numpy as np
import pandas as pd

# ── Build features from fact_content_query_90d (prev30 sub-window) ────────────
# We use the query table for prev30/last30 splits.
# Grain guard: per-content context columns repeat on every query row — use ANY_VALUE().

q_q90d = f"""
SELECT
    client_hash_id,
    content_hash_id,

    -- Safe features: prev30 window (days 31-60 back — before the label window)
    LN(1 + ANY_VALUE(impressions_prev30)) AS log_impressions_prev30,
    LN(1 + ANY_VALUE(clicks_prev30))      AS log_clicks_prev30,
    ANY_VALUE(avg_position_prev30)         AS avg_position_prev30,

    -- Label inputs (kept separate — NOT used as features)
    ANY_VALUE(impressions_last30)          AS impressions_last30_label_input,
    ANY_VALUE(impressions_prev30)          AS impressions_prev30_label_input,

    -- Proxy label: declining = last30 dropped > 20% vs prev30
    CASE
        WHEN ANY_VALUE(impressions_prev30) > 0
             AND ANY_VALUE(impressions_last30) < ANY_VALUE(impressions_prev30) * 0.80
        THEN 1 ELSE 0
    END AS is_declining

FROM read_parquet('{Q90D}')
WHERE ANY_VALUE(impressions_prev30) > 0   -- need a baseline to compute label
GROUP BY client_hash_id, content_hash_id
LIMIT 50000
"""

feat_q90d = con.execute(q_q90d).df()
print(f'Query-90d feature frame shape: {feat_q90d.shape}')
print(f'Label rate (is_declining = 1): {feat_q90d["is_declining"].mean():.1%}')
feat_q90d.head(3)

In [ ]:
# ── Add two features from the daily fact (month=2026-03) ─────────────────────
# days_with_gsc: how many days in March 2026 had ≥1 impression (visibility regularity)
# gsc_avg_position_month: average daily position over the month

q_daily = f"""
SELECT
    client_hash_id,
    content_hash_id,
    COUNT(DISTINCT report_date) FILTER (WHERE gsc_impressions > 0) AS days_with_gsc,
    AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END)  AS gsc_avg_position_month
FROM read_parquet('{FACT}')
WHERE gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
"""

feat_daily = con.execute(q_daily).df()
print(f'Daily fact feature frame shape: {feat_daily.shape}')
feat_daily.head(3)

In [ ]:
# ── Join into the final feature frame ────────────────────────────────────────
feat = feat_q90d.merge(
    feat_daily,
    on=['client_hash_id', 'content_hash_id'],
    how='inner'
).dropna(subset=['log_impressions_prev30', 'log_clicks_prev30',
                  'avg_position_prev30', 'days_with_gsc'])

feature_cols = [
    'log_impressions_prev30',
    'log_clicks_prev30',
    'avg_position_prev30',
    'days_with_gsc',
    'gsc_avg_position_month',
]

print(f'Final feature frame shape: {feat.shape}')
print(f'Label rate (is_declining=1): {feat["is_declining"].mean():.1%}')
print()
print('Feature summary:')
feat[feature_cols].describe().round(3)

### 3b. The leakage trap — watch the score jump, then delete the leak

Here we deliberately add `impressions_last30` as a 6th feature — the numerator of the declining label. AUC should jump toward perfect. Then we remove it and report the honest number.

> **The lesson from notebook 02, performed on real warehouse data:** any column that overlaps with the label's computation window will make the model look nearly perfect. That performance is not real — the model is reading the answer. Delete it before a single line of model code runs.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

# Add the leaked column: log1p(impressions_last30) — the label's numerator
feat['log_impressions_last30_LEAK'] = np.log1p(feat['impressions_last30_label_input'])

y = feat['is_declining']
clf = Pipeline([('scaler', StandardScaler()),
                ('lr', LogisticRegression(max_iter=1000))])

# Score WITH the leak
X_leaked = feat[feature_cols + ['log_impressions_last30_LEAK']]
auc_leaked = cross_val_score(clf, X_leaked, y, cv=5, scoring='roc_auc').mean()
print(f'[LEAKED  ] ROC-AUC with impressions_last30 included : {auc_leaked:.4f}  ← suspiciously high')

# Score WITHOUT the leak (honest)
X_honest = feat[feature_cols]
auc_honest = cross_val_score(clf, X_honest, y, cv=5, scoring='roc_auc').mean()
print(f'[HONEST  ] ROC-AUC after removing the leak         : {auc_honest:.4f}  ← real signal')
print()
print(f'AUC gap caused by leakage: {auc_leaked - auc_honest:+.4f}')
print()
print('Conclusion: log_impressions_last30_LEAK is DELETED.')
print('The honest ROC-AUC above is the number we carry forward.')

# Remove the leaked column — it must not appear in any downstream work
feat.drop(columns=['log_impressions_last30_LEAK',
                    'impressions_last30_label_input',
                    'impressions_prev30_label_input'], inplace=True)

In [ ]:
# Confirm the final feature set — no label-window columns remain
print('Final feature set (safe — no label-window overlap):')
for f in feature_cols:
    print(f'  ✓  {f}')
print()
print('Excluded (leakage or IDs):')
for e in ['impressions_last30', 'clicks_last30', 'avg_position_last30',
           'client_hash_id', 'content_hash_id']:
    print(f'  ✗  {e}')
print()
print('Feature columns present in feat:', list(feat.columns))

---
## 4. Data limits — one named limitation

**Unbalanced panel depth makes cross-client normalisation necessary.**

The `fact_content_daily_performance` panel is unbalanced: per-client history depth ranges from roughly 3 to 17 months depending on each client's `gsc_data_start` (stored in `dim_clients`). In `month=2026-03`, clients who joined the platform after mid-2025 may have fewer than 30 days of measurable prior-period history — meaning their `impressions_prev30` in `fact_content_query_90d` reflects a shorter baseline rather than a true 30-day trailing window. A model trained on raw cross-client values will conflate "new client, limited history" with "declining page". Any cross-client model must normalise per-client, use per-client baseline ratios, or filter to clients with `gsc_data_start ≤ 2025-08` before defining the label. This limitation is invisible in the per-row data; it requires a join to `dim_clients`.

In [ ]:
# Illustrate: count days with GSC data per client in month=2026-03
# Clients with far fewer distinct days than the 31-day month are short-history clients.

q_panel = f"""
SELECT
    client_hash_id,
    COUNT(DISTINCT report_date)                               AS distinct_days_in_month,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)        AS gsc_available_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)        AS ga4_available_rows
FROM read_parquet('{FACT}')
GROUP BY client_hash_id
ORDER BY distinct_days_in_month ASC
LIMIT 10
"""

result_panel = con.execute(q_panel).df()
print('Ten clients with fewest distinct days in month=2026-03 (bottom of the unbalanced panel):')
print(result_panel.to_string(index=False))
print()
print('Limitation confirmed: clients with < 31 distinct days have short or uneven history.')
print('Per-client normalisation is required before cross-client modelling.')

---
## 5. Self-check

Before submitting, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Three verification queries with visible outputs — grain (0 rows), counts+dates, IS TRUE availability
- [x] Five features, each with an "available when?" line in the table above
- [x] The deliberate-leak experiment is shown, the AUC gap is quantified, the column is deleted, honest number kept
- [x] One named limitation stated in plain words and illustrated with a query
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.